# 資料分析

1. 資料讀取
- 載入 `data/df_clean.csv`
- 確認欄位與資料筆數

2. 探索性分析（EDA）

    2.1 基本統計
    - 計算 `Duration (minutes)` 的 mean、median、std、min、max
    - 輸出統計表格

    2.2 各災害種類的 Duration 統計
    - 以 `Cause` 為分組依據
    - 計算各類別的 mean、median、std、min、max
    - 輸出統計表格

    2.3 Cause 分布統計
    - 計算各 Cause 類別的筆數與佔比
    - 繪製長條圖

    2.4 Duration 直方圖
    - 以線性尺度繪製 Duration 直方圖，觀察主體分布
    - 以 log 尺度繪製 Duration 直方圖，觀察尾部行為

3. CCDF 計算與繪製
- 計算經驗 CCDF：對 Duration 排序後計算 P(X > x)
- 以 log-log 尺度繪製 CCDF
- 此圖為後續擬合比較的基準圖

4. Log-normal 分布擬合

    4.1 MLE 估計 μ、σ
    - 對全體資料（Duration > 0）使用 `scipy.stats.lognorm.fit` 進行 MLE 估計
    - 輸出估計參數 μ、σ

    4.2 CCDF 比較圖
    - 在 3. 的 CCDF 圖上疊加 log-normal 擬合曲線
    - 視覺觀察擬合曲線與實際資料的偏離位置，初步判斷尾部開始不符合的區間

5. Log-normal 有效範圍搜尋

    5.1 Binary search + KS test
    - 搜尋範圍：`[log(duration 最小值), log(duration 最大值)]`（log 空間）
    - 每次取中點 `mid`，對 `duration ≤ exp(mid)` 的資料進行 KS test
        - p > 0.10 → 符合 log-normal，往右搜尋（`low = mid`）
        - p ≤ 0.10 → 不符合，往左搜尋（`high = mid`）
    - 收斂條件：`high - low < 1e-3`（log 空間容忍值）
    - 確保每次 KS test 的樣本數 ≥ 30

    5.2 截斷點結果
    - 輸出截斷點 `x_cutoff`（單位：分鐘）
    - 統計截斷點以下的資料筆數與佔比
    - 統計超過截斷點的資料筆數與佔比
    - 在 CCDF 圖上標示截斷點位置

6. 結論

    6.1 有效範圍內的 Log-normal 參數
    - 對截斷點以下的資料重新進行 MLE 估計，輸出最終的 μ、σ
    - 此為後續模擬所使用的 log-normal 參數

    6.2 超過截斷點的事件統計
    - 列出超過截斷點的事件筆數、佔比、對應的 Cause 分布
    - 說明這些事件在模擬中將被設定為 10 天（14,400 分鐘）

# 1. 資料讀取

In [1]:
import pandas as pd
import os

DATA_DIR = "data"

# 載入清理後的資料
df = pd.read_csv(os.path.join(DATA_DIR, "df_clean.csv"))

# 確認欄位與資料筆數
print(f"資料筆數：{len(df)}")
print(f"欄位：{df.columns.tolist()}")
df.head()

資料筆數：9532
欄位：['Name', 'Voltage (kV)', 'Duration (minutes)', 'Outage Type', 'Cause', 'year']


,Name,Voltage (kV),Duration (minutes),Outage Type,Cause,year
0,Creston-Bell No 1 115kV line,115.0,1,Auto,Unknown,2014
1,Creston-Egypt section of Creston-Bell No 1 115...,115.0,1,Auto,Unknown,2014
2,Egypt tap to Creston-Bell No 1 115kV line,115.0,1,Auto,Unknown,2014
3,Egypt-Larene section of Creston-Bell No 1 115k...,115.0,1,Auto,Unknown,2014
4,Larene tap to Creston-Bell No 1 115kV line,115.0,1,Auto,Unknown,2014


# 2. 探索性分析（EDA）

## 2.1 基本統計資料

* 紀錄停電時間的 "平均值", "中位數", "標準差", "最小值", "最大值" 等資訊

* 可以看到中位數遠小於平均值的重尾現象，也就是具有極端事件影響導致平均復電時間較長

In [3]:
# 2.1 基本統計
stats = df["Duration (minutes)"].agg(["mean", "median", "std", "min", "max"])
stats.index = ["Mean", "Median", "Std", "Min", "Max"]
stats.to_frame("Duration (minutes)").round(2)

,Duration (minutes)
Mean,1143.23
Median,93.00
Std,14765.69
Min,1.00
Max,967721.00
